# Problem 8 — A Two-Half-Plane Torch Model

Build one inference-only region detector from fresh geometry. Parts 8.2–8.4 consume the module from Part 8.1.


In [ ]:
import torch
from torch import nn

torch.set_default_dtype(torch.float64)  # C6 course register


## Part 8.1 (5 points)

**Type:** programming · **Difficulty: intro** · **Answer form: code** · **Concepts: threshold-activation, nn-module**  
**Flag:** Coding allowed.

Complete the exact My_CamelCase module Inclusive_Step. Its forward method returns 1 where x is greater than or equal to zero and 0 otherwise. The result must preserve x.dtype and x.shape, and the activation must fire at exactly zero.


In [ ]:
class Inclusive_Step(nn.Module):
    def forward(self, x):
        return (x >= 0).to(dtype=x.dtype)


_step_probe = torch.tensor([-0.25, 0.0, 0.75], dtype=torch.float64)
_step_result = Inclusive_Step()(_step_probe)
assert _step_result.shape == _step_probe.shape
assert _step_result.dtype == _step_probe.dtype

In [ ]:
ANSWER = 2.0
assert torch.isclose(torch.tensor(ANSWER), _step_result.sum(), atol=0, rtol=0)

## Part 8.2 (5 points)

**Type:** programming · **Difficulty: core** · **Answer form: code** · **Concepts: decision-boundaries-geometric, manual-weights**  
**Flag:** Coding allowed.

The target region is the intersection of these fresh half-planes:

- 5x1 - 2x2 - 11 >= 0
- -7x1 - 3x2 + 13 >= 0

Create the exact float64 tensors plane_weight with shape (2, 2) and plane_bias with shape (2,) so each affine output is the corresponding left-hand score. Instantiate Half_Plane_Pair as half_plane_pair. Its forward pass must return the two inclusive 0/1 decisions for every input row.


In [ ]:
class Half_Plane_Pair(nn.Module):
    def __init__(self, weight, bias):
        super().__init__()
        self.affine = nn.Linear(2, 2)
        with torch.no_grad():
            self.affine.weight.copy_(weight)
            self.affine.bias.copy_(bias)
        self.gate = Inclusive_Step()

    def forward(self, x):
        return self.gate(self.affine(x))


plane_weight = torch.tensor([[5.0, -2.0], [-7.0, -3.0]], dtype=torch.float64)
plane_bias = torch.tensor([-11.0, 13.0], dtype=torch.float64)
half_plane_pair = Half_Plane_Pair(plane_weight, plane_bias)
assert plane_weight.shape == (2, 2)
assert plane_bias.shape == (2,)

In [ ]:
affine_coefficient_sum = plane_weight.sum() + plane_bias.sum()
ANSWER = -5.0
assert torch.isclose(torch.tensor(ANSWER), affine_coefficient_sum, atol=0, rtol=0)

## Part 8.3 (5 points)

**Type:** programming · **Difficulty: advanced** · **Answer form: code** · **Concepts: mlp-architecture, custom-layers**  
**Flag:** Coding allowed.

Complete the exact My_CamelCase module Channel_Region. It must compose the half_plane_pair from Part 8.2 with a 2-to-1 nn.Linear readout and Inclusive_Step from Part 8.1. Hand-set the readout so the final output is 1 exactly when both half-plane decisions are 1, including points on either boundary. Store the model as region_model. Run all probes in one float64 batch inside torch.inference_mode(); the supplied assertion is part of the contract.


In [ ]:
class Channel_Region(nn.Module):
    def __init__(self, pair):
        super().__init__()
        self.pair = pair
        self.readout = nn.Linear(2, 1)
        with torch.no_grad():
            self.readout.weight.copy_(torch.tensor([[1.0, 1.0]], dtype=torch.float64))
            self.readout.bias.copy_(torch.tensor([-1.5], dtype=torch.float64))
        self.gate = Inclusive_Step()

    def forward(self, x):
        return self.gate(self.readout(self.pair(x)))


region_model = Channel_Region(half_plane_pair)
region_model.eval()
region_probes = torch.tensor([
    [3.0, 1.0],
    [2.0, -1.0],
    [0.0, 0.0],
    [1.0, -3.0],
], dtype=torch.float64)
with torch.inference_mode():
    region_membership = region_model(region_probes)

expected_membership = torch.tensor([[0.0], [1.0], [0.0], [1.0]], dtype=torch.float64)
assert torch.equal(region_membership, expected_membership)

In [ ]:
ANSWER = 2.0
assert torch.isclose(torch.tensor(ANSWER), region_membership.sum(), atol=0, rtol=0)

## Part 8.4 (5 points)

**Type:** programming · **Difficulty: core** · **Answer form: code** · **Concepts: parameter-counting**  
**Flag:** Coding allowed.  
**Ban (zero points for this part):** Do not use numel, nelement, parameters(), named_parameters(), state_dict(), shape-product utilities, or any disguised counting helper.

By literal arithmetic only, count every scalar nn.Parameter in region_model from Part 8.3. Include weights and biases in its 2-to-2 half-plane affine map and 2-to-1 readout; Inclusive_Step contributes none. Assign the integer literal to parameter_total and show the layer-by-layer arithmetic in a comment.


In [ ]:
# Literal arithmetic: (2 * 2 weights + 2 biases) + (2 * 1 weights + 1 bias)
parameter_total = (2 * 2 + 2) + (2 * 1 + 1)
assert isinstance(parameter_total, int)

In [ ]:
ANSWER = 9
assert torch.isclose(torch.tensor(float(ANSWER)), torch.tensor(float(parameter_total)), atol=0, rtol=0)